<a href="https://colab.research.google.com/github/aryankumar29082006-dotcom/agentloop-/blob/main/recipe_agent_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About
This colab is an introduction to a RAG based ReAct agent that looks at a recipe book, reasons the best course of action and generates a response.

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to aistudio.google.com
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [2]:
# Quiet ensures your UI is not full of logs.
# Install the Gemini SDK
!pip install -q google-genai

print("Done.")

Done.


In [3]:
# Create a database of student documents.

DOCUMENTS = {
    "sports": """
    Students can share documents about cricket, football and basketball.
    They can discuss match results, player performances and tournament records.
    Documents may include team statistics and training techniques.
    Students can also share information about upcoming sporting events.
    """,

    "manga": """
    Students can share documents about popular manga series.
    They can discuss characters, storylines and different manga genres.
    Documents may include manga recommendations and author information.
    Students can also compare different manga series.
    """,

    "food": """
    Students can share documents about different cuisines.
    They can discuss recipes, cooking techniques and traditional dishes.
    Documents may include nutritional information and food cultures.
    Students can also share their favourite restaurants and food recommendations.
    """,

    "music": """
    Students can share documents about different music genres.
    They can discuss artists, albums and musical instruments.
    Documents may include music recommendations and concert information.
    Students can also share their favourite songs and playlists.
    """
}


def search_documents(query: str) -> str:
    """Search the document collection for relevant paragraphs."""

    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(
            1 for word in words if word in content.lower()
        )

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    output = []

    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")

    return "\n\n".join(output)


system_instruction = """
You are a student document assistant.
Answer questions using the information retrieved
from the available documents.
If the documents do not contain the answer,
clearly state that the information is unavailable.
Do not invent information.
"""

# Quick test
print(search_documents("students share documents"))

[Source: sports]

    Students can share documents about cricket, football and basketball.
    They can discuss match results, player performances and tournament records.
    Documents may include team statistics and training techniques.
    Students can also share information about upcoming sporting events.
    

[Source: music]

    Students can share documents about different music genres.
    They can discuss artists, albums and musical instruments.
    Documents may include music recommendations and concert information.
    Students can also share their favourite songs and playlists.
    


In [4]:

# @title Define a tool

def search_documents(query: str) -> str:
    """Search the document collection for relevant information."""

    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(
            1 for word in words if word in content.lower()
        )

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []

    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")

    return "\n\n".join(output)


# Quick test
print(search_documents("students share documents"))


[Source: sports]

    Students can share documents about cricket, football and basketball.
    They can discuss match results, player performances and tournament records.
    Documents may include team statistics and training techniques.
    Students can also share information about upcoming sporting events.
    

[Source: music]

    Students can share documents about different music genres.
    They can discuss artists, albums and musical instruments.
    Documents may include music recommendations and concert information.
    Students can also share their favourite songs and playlists.
    


In [5]:
# @title Ask gemini to search for something

from google import genai
from google.genai import types

client = genai.Client()

MODEL_NAME = "gemini-3.5-flash-lite"

config = types.GenerateContentConfig(
    system_instruction="""
    You are a student document assistant.
    Answer questions using the information retrieved
    from the available documents.
    If the documents do not contain the answer,
    clearly state that the information is unavailable.
    Do not invent information.
    """
)

print("Model ready:", MODEL_NAME)


Model ready: gemini-3.5-flash-lite


In [6]:

import time
from google import genai
from google.genai import types

# Initialize Gemini client
client = genai.Client()

# Define document search tool
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the student document collection for relevant information.",
            parameters={
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"],
            },
        )
    ]
)

# Configure the model
config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction=system_instruction,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
)


# Retry function for temporary server errors
def send_with_retry(contents, max_retries=5):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )

        except Exception as e:
            error = str(e)

            if ("503" in error or "UNAVAILABLE" in error):
                if attempt < max_retries - 1:
                    wait = 30 * (attempt + 1)
                    print(f"Server busy. Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    raise
            else:
                raise


# Run the document assistant
def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Preserve the model's response
        contents.append(model_content)

        calls = [
            p.function_call
            for p in model_content.parts
            if p.function_call
        ]

        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Execute requested tool calls
        result_parts = []

        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")

            result = search_documents(**args)

            print(f"Result preview: {str(result)[:100]}...")

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        contents.append(
            types.Content(role="user", parts=result_parts)
        )

    return "(Stopped: hit max_steps)"


# Test the document assistant
_ = run_agent("What topics are students supposed to share documents on?")


Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'share documents topics'})
Result preview: [Source: sports]

    Students can share documents about cricket, football and basketball.
    They ...

Final answer:
Based on the documents, students are supposed to share documents on the following topics:

* **Sports:** Cricket, football, and basketball (including match results, player performances, tournament records, team statistics, training techniques, and upcoming sporting events).
* **Music:** Different music genres (including artists, albums, musical instruments, music recommendations, concert information, and favorite songs and playlists).


In [7]:
questions = [
    "What topics are students supposed to share documents on?",
    "What information can students share about sports?",
    "What can students discuss about manga?",
    "What topics are covered in the music documents?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)
    print("\n" + "=" * 60)
    run_agent(q)

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'share documents topics'})
Result preview: [Source: sports]

    Students can share documents about cricket, football and basketball.
    They ...

Final answer:
Based on the documents, students are supposed to share documents on two main topics:

1. **Sports**: Cricket, football, and basketball (including match results, player performances, tournament records, team statistics, training techniques, and upcoming sporting events).
2. **Music**: Different music genres (including artists, albums, musical instruments, music recommendations, concert information, favorite songs, and playlists).

Question: What information can students share about sports?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'sports'})
Result preview: No relevant documents found for that query....
Step 2

# Next steps:
* Save a copy of this code to your GitHub repo.
* Create a tool to recommend ingredient substitutions for recipes.
* Push the new tool to your colab and v2 to the same GitHub repo.